Project Drishti · Any data. Any domain. One grammar.

Copyright (c) 2026 Ashutosh Sinha <ajsinha@gmail.com>.
All rights reserved.

PROPRIETARY AND CONFIDENTIAL.

This file is the confidential and proprietary property of Ashutosh Sinha.
Unauthorised copying, use, modification, distribution or disclosure of this
file, via any medium, is strictly prohibited except with the express prior
written permission of the copyright holder.

See the LICENSE file in the root of this repository for the full terms.

# 08 · A table of the in-memory lake

Some feeds are held in memory only (a vendor's contract may forbid keeping them on disk: IN_MEMORY_LAKE_GUIDE.md). Their tables live on the node that owns the store, and Flight is how a program reads them: `form: lake`, the lake and the table named as the lake configuration names them. The plan's endpoint points at the owner node.

**Keep it in memory here too:** the contract that keeps the feed off the server's disks applies to your copy. This notebook writes nothing to disk.

*Read only:* this notebook only reads. Flight refuses every call that would write (`DoPut`, `DoExchange`), and nothing here asks for one.

**Setup.** `flight_setup.py` (beside this notebook) finds the one-file client `clients/python/drishti_flight.py` and reads the address, the token and the CA from the environment: `DRISHTI_FLIGHT_URL`, `DRISHTI_TOKEN` (or `DRISHTI_TOKEN_FILE`) and `DRISHTI_FLIGHT_CA`. No token is ever written in a notebook.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))     # run Jupyter from examples/clients/python, where flight_setup.py is
from flight_setup import connect, meta, FlightRefused, SCOPE, DATE, STREAM_WAIT

## Which lake tables you may read

The catalogue lists every lake table the token may read. `DRISHTI_EXAMPLE_MEM_LAKE` (default `news`, the in-memory store of tutorial 31) and `DRISHTI_EXAMPLE_MEM_KIND` choose one; otherwise the first table of that lake is taken.

In [ ]:
import os

f = connect()
lake_tables = [e["request"] for e in f.catalogue() if e.get("form") == "lake" and isinstance(e.get("request"), dict)]
for t in lake_tables:
    print(t.get("lake"), "/", t.get("kind"))
LAKE = os.environ.get("DRISHTI_EXAMPLE_MEM_LAKE", "news")
KIND = os.environ.get("DRISHTI_EXAMPLE_MEM_KIND") or next((t["kind"] for t in lake_tables if t.get("lake") == LAKE), None)
print("reading:", LAKE, "/", KIND)

## Where it is held

The plan names, for each part, the nodes that can answer it: for an in-memory table, the owner first, then a peer that holds a copy if there is one.

In [ ]:
request = f.request("lake", {"lake": LAKE, "kind": KIND})
info = f.plan(request)
for i, ep in enumerate(info.endpoints):
    print("part", i, [loc.uri.decode() if isinstance(loc.uri, bytes) else str(loc.uri) for loc in ep.locations])

## Read it

The same keys as a kind (`paths`, `where`); the rows are the table's on its latest date.

In [ ]:
table = f.lake(LAKE, KIND)
print(table.num_rows, "rows;", table.num_columns, "columns")
table.slice(0, 10).to_pandas()

## New rows as they arrive (optional)

A table with a whole-number column that grows with each append (a *cursor*) can be followed live, as in notebook 06: set `DRISHTI_EXAMPLE_MEM_CURSOR` to that column's name.

In [ ]:
CURSOR = os.environ.get("DRISHTI_EXAMPLE_MEM_CURSOR")
if CURSOR:
    live = connect(timeout=STREAM_WAIT)
    stream = live.stream(live.request("lake", {"lake": LAKE, "kind": KIND, "cursor": CURSOR}))
    try:
        for seq, batch in stream:
            print(f"up to {CURSOR} {seq}: {batch.num_rows} rows")
            break
    except FlightRefused as e:
        if e.code:
            raise
        print(f"no new rows within {STREAM_WAIT:.0f} seconds")
    finally:
        stream.close()
        live.close()
else:
    print("no cursor column set: snapshot only")

In [ ]:
f.close()